Integrating all code snipppets to get one script as I lost the overview what is up to date to be honest .. to many

I've run the naming of the clustering (TF-IDF) and the clustering with the answers again after handling the noise because I think we forgot that in the parted script before :)

In [19]:
# Imports
import re
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
import umap
import hdbscan
import random
import torch
import collections
import copy
import nltk
from time import sleep
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag, word_tokenize
import spacy

In [20]:
import spacy.cli
spacy.cli.download("de_core_news_sm")

✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [21]:
#Downloads 
nltk.download("punkt")
nltk.download("averaged_perceptron_tagger")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("stopwords")
nlp = spacy.load("de_core_news_sm")
# Evtl. in Terminal auszuführen: "python -m spacy download de_core_news_sm" oder spacy.cli.download("de_core_news_sm") import spacy.cli


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cdoering\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\cdoering\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\cdoering\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\cdoering\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\cdoering\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
#Setup Mistral API
from mistralai import Mistral
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
modelAI = "open-mistral-nemo"
client = Mistral(api_key=api_key)

Load the competency questions for all documents (our filtered data set) and generate clusters with UMAP Dimensionality Reduction, Sentence Embedding and HDBSCAN

In [23]:
# Load competency questions from file
with open("competency_questions_output/competency_questions_all_documents.txt", "r", encoding="utf-8") as f:
    content = f.read()

# Extract questions and sources as pairs
pattern = r'\*\*Frage:\*\*\s*(.+?)\s*\*\*Quelle:\*\*\s*(.+?)(?=\n\d+\.\Z|\n\d+\.|\Z)'
entries = re.findall(pattern, content, re.DOTALL)

# Separate question and source for processing
questions_only = [q.strip() for q, s in entries]
sources_only = [s.strip() for q, s in entries]

# Deduplicate questions (with mapping)
seen = set()
questions, sources = [], []
for q, s in zip(questions_only, sources_only):
    if q not in seen:
        seen.add(q)
        questions.append(q)
        sources.append(s)

print(f"✅ Loaded {len(questions)} questions.")

✅ Loaded 55108 questions.


In [ ]:
#HDBSACN
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

#Sentence Embedding (SBERT)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(questions)

#Dimensionality Reduction (UMAP, optional but good for HDBSCAN & visualization)
umap_reducer = umap.UMAP(n_neighbors=20, n_components=2, metric='cosine', random_state=42)
X_umap = umap_reducer.fit_transform(embeddings)

#HDBSCAN clustering (as found in the Ntropy article)
clusterer = hdbscan.HDBSCAN(min_cluster_size=15, min_samples=7, metric='euclidean', prediction_data=True)
labels = clusterer.fit_predict(X_umap)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"📌 Number of clusters found by HDBSCAN: {n_clusters}")

n_noise = np.sum(labels == -1)
print(f"🧩 Anzahl der CQs im Noise-Cluster (-1): {n_noise}")


#Evaluation (excluding outliers)
valid_mask = labels != -1
sil_score = silhouette_score(X_umap[valid_mask], labels[valid_mask])
print(f"📈 Silhouette Score (without outliers): {sil_score:.4f}")

#Visualization (t-SNE)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_2d = tsne.fit_transform(X_umap)

#Create the Plot
plt.figure(figsize=(10, 6))
palette = sns.color_palette('husl', len(set(labels)))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=labels, palette=palette, legend='full')

#Trim down the Legend to only 20 Colours being showen
handles, labels_ = plt.gca().get_legend_handles_labels()
plt.legend(handles[:21], labels_[:21], title="Cluster (max. 20)", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.title("t-SNE visualization of clustered questions")
plt.tight_layout()
plt.show()

# Save Basic Cluster with emebeddings for Noise Handling
clustered_cqs = {}
for q, l, e in zip(questions, labels, embeddings):
    entry = {"cq": q, "embedding": e.tolist()}
    clustered_cqs.setdefault(f"Cluster {l}", []).append(entry)

with open("clustered_questions/clustered_CQs_with_embeddings_new.json", "w", encoding="utf-8") as f:
    json.dump(clustered_cqs, f, ensure_ascii=False, indent=4)

Use noise triage to handle noise in clusters 

19730/55108 competency questions were identified as noise

In [ ]:
#Handle Noise (Triage) 
data = clustered_cqs
noise_cl = data.get("Cluster -1", [])
print(f"Noise CQs: {len(noise_cl)} / {sum(len(v) for v in data.values())} total")

PROMOTE_THRESHOLD = 0.15
REVIEW_THRESHOLD = 0.20

noise_embeddings = np.array([x["embedding"] for x in noise_cl])
core_embeddings = []
core_index_to_cluster = {}
c_idx = 0
for cname, cqs in data.items():
    if cname == "Cluster -1": continue
    for cq in cqs:
        core_embeddings.append(cq["embedding"])
        core_index_to_cluster[c_idx] = cname
        c_idx += 1
core_embeddings = np.array(core_embeddings)

from sklearn.metrics.pairwise import cosine_distances
dist_matrix = cosine_distances(noise_embeddings, core_embeddings)

# Get the minimum distance and index of closest core for each noise CQ
d_min = dist_matrix.min(axis=1)
nearest_idx = dist_matrix.argmin(axis=1)

triaged = {"promote": [], "review": [], "leave": []}
for i, cq in enumerate(noise_cl):
    distance = d_min[i]
    nearest_cluster = core_index_to_cluster[nearest_idx[i]]
    tag = "promote" if distance <= PROMOTE_THRESHOLD else "review" if distance <= REVIEW_THRESHOLD else "leave"
    triaged[tag].append({"cq": cq["cq"], "embedding": cq["embedding"], "d_min": float(distance), "nearest_cluster": nearest_cluster})

with open("competency_questions_output/noise_triage_new.json", "w", encoding="utf-8") as f:
    json.dump(triaged, f, ensure_ascii=False, indent=2)


#Update clustered data with triaged noise
updated_data = {k: copy.deepcopy(v) for k, v in data.items() if k != "Cluster -1"}

#Add promoted and review noise CQs to their nearest clusters and create a new "Cluster -1" for those that are left
for tag in ["promote", "review"]:
    for item in triaged[tag]:
        item["from_noise"] = tag
        updated_data[item["nearest_cluster"]].append(item)
updated_data["Cluster -1"] = [{**item, "from_noise": "leave"} for item in triaged["leave"]]

ordered_data = dict(sorted(updated_data.items(), key=lambda x: (x[0] != "Cluster -1", int(x[0].split()[-1]) if x[0].startswith("Cluster ") else float('inf'))))

with open("clustered_questions/clustered_CQs_with_promoted_and_reviewed_noise_new.json", "w", encoding="utf-8") as f:
    json.dump(ordered_data, f, ensure_ascii=False, indent=2)
 

print(f"Promoted: {len(triaged['promote'])}, Reviewed: {len(triaged['review'])}, Left: {len(triaged['leave'])}")
print(f"✅ Updated clustered data with triaged noise. Total clusters: {len(ordered_data)}")   


#Save noise handled clusters also in readable format
clustered_questions_only = {}

for cluster_name, items in ordered_data.items():
    clustered_questions_only[cluster_name] = [entry["cq"] for entry in items if "cq" in entry]

with open("clustered_questions/clustered_questions_only_new.json", "w", encoding="utf-8") as f:
    json.dump(clustered_questions_only, f, ensure_ascii=False, indent=2)

print("✅ Saved readable cluster overview to 'clustered_questions_only_new.json'.")


NameError: name 'clustered_cqs' is not defined

Use TF-IDF and several wordlist downloads to label the noise-handled clusters 

In [ ]:
#Create a folder to store the needed label data for our Dashboard
os.makedirs("dashboard_data", exist_ok=True)

In [ ]:
#Load Stopwords (standard of given packages + our own "legal" additions)
base_stopwords = stopwords.words('german')
own_additions = [
    "welche", "hat", "daher", "damit", "ob", "um", "wie", "wer", "was", "wann", "wo", "wurde",
    "liegt", "besteht", "enthält", "stellt", "erfüllt", "bezieht", "basiert", "bestimmen", "bewerten",
    "gilt", "spielt", "tritt", "verlangt", "aufweist", "führt", "genannt", "verweist",
    "rechtsgrundlage", "rechtsfolge", "recht", "rechtlich", "rechtliche", "rechtlicher", "rechtlichen",
    "gesetzlich", "gerichtlich", "rechtsprechung", "rechtsprechungen",
    "anspruch", "ansprüche", "anspruchs", "ansprüchen", "anspruchsgrundlage", "anspruchsgrundlagen",
    "forderung", "forderungen", "entscheidung", "feststellung", "anwendung", "voraussetzungen",
    "urteils", "urteil", "klage", "verfahren", "geltend", "geltendmachung",
    "bgb", "zpo", "hgb", "egbgb", "prodhaftg", "hoai", "vglo", "satz", "abs", "nr", "art", "nach", "gemäß",
    "auch", "nicht", "sowie", "jedoch", "nur", "noch", "bereits", "alle", "mehr", "weniger", "einschließlich",
    "beklagten", "kläger", "klägerin", "beklagte", "parteien", "person", "personen",
    "vertrag", "vertrages", "vertrags", "vereinbarung", "vereinbart", "vereinbarten", "regelung", "rahmen",
    "pflichten", "rechte", "bedingungen", "bestimmung", "bestimmungen",
    "müssen", "dürfen", "sollen", "können", "dürfte", "wird", "sind", "sein", "verpflichtet", "unwirksam", "wirksam",
    "rolle", "bedeutung", "wichtige", "wichtiger", "insbesondere", "einschlägig", "wesentliche", "wesentlichen",
    "warum", "auf", "hinblick", "betreffend", "weiteren", "anklageschrift", "abweichen", "geltung", "geltungsbereich",
    "gericht", "gerichtliche", "gerichtlicher", "gerichtlichen", "rechtskraft", "zpo", 
    "satz", "abs", "nr", "art", "nach", "gemäß", "rechtsfolge", "satzung"
]
stopwords_de = list(set(base_stopwords + own_additions))

#Synonyms, Lemmatization & POS-Filter
synonym_map = {"vergütung": ["entlohnung", "honorar", "bezahlung"]}

def process_text(text, mapping):
    for target, syns in mapping.items():
        for syn in syns:
            text = text.replace(syn, target)
    
    doc = nlp(text.lower())
    lemmatized_nouns = [
        token.lemma_ for token in doc
        if token.pos_ == "NOUN" and token.lemma_ not in stopwords_de and len(token.lemma_) > 2 and not token.is_digit
    ]
    return " ".join(lemmatized_nouns)

#Prepare the Data for TF-IDF
all_questions, all_labels = [], []
for cname, cqs in ordered_data.items():
    if cname == "Cluster -1":
        continue
    for cq in cqs:
        all_questions.append(cq["cq"])
        all_labels.append(cname)

processed_questions = [process_text(q, synonym_map) for q in all_questions]

#Calculation for TF-IDF
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(processed_questions)
feature_names = np.array(vectorizer.get_feature_names_out())

#Find Top Keywords per Cluster
cluster_to_questions = collections.defaultdict(list)
for q, label in zip(processed_questions, all_labels):
    cluster_to_questions[label].append(q)

print(" Top TF-IDF Keywords per Cluster:")
cluster_keywords = {}
for cluster_id, texts in cluster_to_questions.items():
    matrix = vectorizer.transform(texts).mean(axis=0).A1
    top_keywords = feature_names[matrix.argsort()[-7:][::-1]]
    cluster_keywords[cluster_id] = top_keywords.tolist()
    print(f" {cluster_id} (n={len(texts)}):\nTop Keywords: {', '.join(top_keywords)}")

#Save the cluster labels as JSON for later use in our Dashboard
with open("dashboard_data/cluster_keywords.json", "w", encoding="utf-8") as f:
    json.dump(cluster_keywords, f, ensure_ascii=False, indent=2)

NameError: name 'ordered_data' is not defined

Generate generalized questions for the noise reduced clusters 

In [ ]:
#Load Our Clustered CQs
with open("clustered_questions/clustered_questions_only_new.json", "r", encoding="utf-8") as f:
    ordered_data = json.load(f)

#Generate Generalized Competency Questions (GCQs) from these clusterd CQs
results = {}

system_prompt = """
You are a legal knowledge engineering expert.

Your task is to generate a **generalized competency question (GCQ)** from a list of **clustered German legal competency questions (CQs)**.

---

## Goal:
Create **one single abstract, representative legal question** that captures the core meaning of all input CQs.

## Instructions:
- Output one well-formed, precise, **legally relevant** question in German.
- The question should reflect the **shared semantics** of the cluster without copying specific details.
- Use **legal terminology** such as:
  - “Unter welchen Voraussetzungen…”
  - “Welche rechtliche Bedeutung hat…”
  - “Wer ist verpflichtet…”
- Do not summarize, explain, or list individual CQs. Just output the **generalized question**.
- Output **only the question**, in natural, legal German.

---

## Examples:

### Cluster Example 1:
Input:
- Welche Rechtsgrundlage gilt für die Präklusion einer Aufrechnung aufgrund von Verspätung?
- Welche Rechtsgrundlage gilt für die Zurechnung von Handlungen des Auslieferungsfahrers auf den Frachtführer?
- Welche Rechtsgrundlage gilt für die Haftung bei grob fahrlässig herbeigeführten Arbeitsunfällen?
- Welche Rechtsgrundlage gilt für den Nachweis von grober Fahrlässigkeit bei Arbeitsunfällen?
- Welche Rechtsgrundlage gilt für die Haftung eines Dritten, der ohne Verschulden an einem Schaden beteiligt ist?
- Welche Rechtsgrundlage gilt für die Haftung eines Unternehmens bei Verletzung von Datenschutzbestimmungen?
- Wie ist die Zurechnung von Beratungsfehlern eines Dritten auf einen Verkäufer im Rahmen von Schadensersatzansprüchen zu beurteilen?
- Welche Rechtsgrundlage gilt für die Zurechnung von Beratungsfehlern eines Dritten auf eine Bank?
- Welche Rechtsgrundlage gilt für die Zurechnung von Schäden, die durch den Rücktritt vom Kaufvertrag entstehen?
- Welche Bedeutung hat die Verletzung oder Tötung des Inhabers oder Mitarbeiters eines Unternehmens für den Gewerbebetrieb?
- Welche Rechtsgrundlage gilt für den Ersatz eines Schadens, der auf den Verlust eines Leibgedings zurückzuführen ist?
- Welche Rolle spielt das institutionalisierte Zusammenwirken bei der Zurechnung von Beratungsfehlern?

GCQ:
Unter welchen Voraussetzungen wird ein Verhalten rechtlich einer anderen Person oder Stelle zugerechnet?

---

### Cluster Example 2:
Input:
- Welche Voraussetzungen müssen erfüllt sein, damit Versicherte einen Anspruch auf Überschussbeteiligung durch die Beklagte haben?
- Wie wird die Höhe der Überschussbeteiligung durch die Beklagte bestimmt?
- Welche Rechte haben die Versicherten, wenn die Beklagte den satzungsgemäßen Vorgaben für die Überschussbeteiligung nicht nachkommt?
- Welche Rechtsgrundlage gilt für den Anspruch der Versicherten auf Überschussbeteiligung entsprechend den satzungsgemäßen Vorgaben?
- Welche Rechtsfolge tritt ein, wenn die Beklagte den satzungsgemäßen Vorgaben für die Überschussbeteiligung nicht nachkommt und die Versicherten keine gerichtliche Feststellung begehren?
- Welche Umstände können die Höhe der Entschädigung beeinflussen?

GCQ:
Wann entsteht ein Anspruch auf eine Beteiligung an Überschüssen aus einem Versicherungsverhältnis?

---

### Cluster Example 3:
Input:
- Welche Voraussetzungen müssen erfüllt sein, damit ein Käufer nach § 437 Nr. 3 BGB Schadensersatz für einen mangelbedingten Nutzungsausfall verlangen kann, auch wenn er vom Kaufvertrag zurückgetreten ist?
- Wer ist verpflichtet, Wertersatz für die Nutzung einer Kaufsache zu zahlen, wenn der Käufer vom Kaufvertrag zurücktritt?
- Wie ist der Schadensersatzanspruch des Käufers nach § 437 Nr. 3 BGB im Falle eines Rücktritts vom Kaufvertrag zu berechnen?
- Welche Rolle spielt der § 325 BGB bei der Geltendmachung eines Schadensersatzanspruchs durch den Käufer nach einem Rücktritt vom Kaufvertrag?
- Wie ist der Schadensersatzanspruch des Käufers nach § 437 Nr. 3 BGB zu berechnen, wenn der Käufer die Reparaturkosten für das mangelhafte Fahrzeug nicht selbst tragen muss?
- Welche Bedingungen müssen erfüllt sein, damit der Käufer nach § 437 Nr. 3 BGB Schadensersatz für einen mangelbedingten Nutzungsausfall verlangen kann?
- Welche Rechte hat der Käufer, der aufgrund eines Mangels vom Kaufvertrag zurücktritt und Schadensersatz nach § 437 Nr. 3 BGB beansprucht?
- Wie ist der Schadensersatzanspruch des Käufers nach § 437 Nr. 3 BGB im Falle eines Rücktritts vom Kaufvertrag und einer Herausgabe der mangelhaften Kaufsache zu berechnen?

GCQ:
Unter welchen Bedingungen bleibt ein Anspruch auf Schadensersatz bestehen, wenn ein Vertragsverhältnis rückabgewickelt wird?

"""


for cluster_id, questions in ordered_data.items():
    if cluster_id == "Cluster -1":
        continue  # Skip the Noise Cluster
    print(f"🧠 Generiere GCQ für {cluster_id} mit {len(questions)} Fragen...")
    prompt = system_prompt + "\n\nFragen:\n" + "\n".join(f"- {q}" for q in questions) + "\n\nGeneralisierte Frage:"
    try:
        response = client.chat.complete(
            model=modelAI,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        )
        results[cluster_id] = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Fehler bei Cluster {cluster_id}:", e)
        results[cluster_id] = "Error generating question."
    sleep(3)

with open("competency_questions_output/generalized_questions_new.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Fertig – generalisierte Fragen wurden gespeichert.")

🧠 Generiere GCQ für Cluster 0 mit 37 Fragen...
🧠 Generiere GCQ für Cluster 1 mit 270 Fragen...
🧠 Generiere GCQ für Cluster 2 mit 193 Fragen...
🧠 Generiere GCQ für Cluster 3 mit 54 Fragen...
🧠 Generiere GCQ für Cluster 4 mit 64 Fragen...
🧠 Generiere GCQ für Cluster 5 mit 53 Fragen...
🧠 Generiere GCQ für Cluster 6 mit 111 Fragen...
🧠 Generiere GCQ für Cluster 7 mit 120 Fragen...
🧠 Generiere GCQ für Cluster 8 mit 118 Fragen...
🧠 Generiere GCQ für Cluster 9 mit 37 Fragen...
🧠 Generiere GCQ für Cluster 10 mit 32 Fragen...
🧠 Generiere GCQ für Cluster 11 mit 90 Fragen...
🧠 Generiere GCQ für Cluster 12 mit 16 Fragen...
🧠 Generiere GCQ für Cluster 13 mit 93 Fragen...
🧠 Generiere GCQ für Cluster 14 mit 31 Fragen...
🧠 Generiere GCQ für Cluster 15 mit 192 Fragen...
🧠 Generiere GCQ für Cluster 16 mit 19 Fragen...
🧠 Generiere GCQ für Cluster 17 mit 75 Fragen...
🧠 Generiere GCQ für Cluster 18 mit 45 Fragen...
🧠 Generiere GCQ für Cluster 19 mit 43 Fragen...
🧠 Generiere GCQ für Cluster 20 mit 112 Frage

Include the sources for each question

In [ ]:
#Load the Original CQ-to-Source Mappings
cq_source_map = {}

with open("competency_questions_output/competency_questions_all_documents.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# atch CQ and Source (Frage + Quelle) in Pairs
pattern = r'\*\*Frage:\*\*\s*(.+?)\s*\*\*Quelle:\*\*\s*(.+?)(?=\n\d+\.|\Z)'
matches = re.findall(pattern, raw_text, re.DOTALL)

for question, source in matches:
    cleaned_q = question.strip()
    cleaned_source = source.strip().strip('"').strip()
    cq_source_map[cleaned_q] = cleaned_source

print(f"✅ Extracted {len(cq_source_map)} CQ-to-source pairs.")

#Load Our Clustered Data
with open("clustered_questions/clustered_questions_only_new.json", "r", encoding="utf-8") as f:
    clustered_data = json.load(f)

#Map CQs in Clusters to their Original Sources
clustered_with_sources = {}
missing_sources = []

for cluster_name, cqs in clustered_data.items():
    cluster_entries = []
    for cq_entry in cqs:
        question = cq_entry.strip()
        source = cq_source_map.get(question)
        if not source:
            missing_sources.append(question)
            source = "UNKNOWN"  #Fallback just inncase
        cluster_entries.append({
            "cq": question,
            "quelle": source
        })
    clustered_with_sources[cluster_name] = cluster_entries

#Save to JSON
with open("clustered_questions/clustered_CQs_with_sources.json", "w", encoding="utf-8") as f:
    json.dump(clustered_with_sources, f, ensure_ascii=False, indent=2)

print(f"✅ Saved clustered questions with sources to 'clustered_CQs_with_sources.json'.")
if missing_sources:
    print(f"⚠️ {len(missing_sources)} questions had no matching source.")
    with open("missing_sources_log.txt", "w", encoding="utf-8") as f:
        for q in missing_sources:
            f.write(q + "\n")
    print("⚠️ Missing questions logged in 'missing_sources_log.txt'.")

✅ Extracted 55108 CQ-to-source pairs.
✅ Saved clustered questions with sources to 'clustered_CQs_with_sources.json'.
